# f696 八项旧输入的匹配好坏区间对照

八项因子与标准化口径在读取评价区间前固定；Qlib来源和账户均逐对象核对。此处只生成描述性研究假说，不选择交易阈值。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R

root=Path.cwd();card=root/"research/stage1_f696_eight_input_interval_contrast_20261024"
plan_path=card/"plan.json";plan=json.loads(plan_path.read_text());dep=plan["dependencies"]
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan["frozen"] and all(v==0 for v in plan["budget"].values())
drift=root/dep["input_drift_card"];interval=root/dep["interval_card"]
assert sha(drift/"plan.json")==dep["input_drift_plan_sha256"] and sha(drift/"result.json")==dep["input_drift_result_sha256"]
assert sha(interval/"plan.json")==dep["interval_plan_sha256"] and sha(interval/"result.json")==dep["interval_result_sha256"]
dp=json.loads((drift/"plan.json").read_text());ip=json.loads((interval/"plan.json").read_text())
assert json.loads((drift/"result.json").read_text())["held_sessions"]==944
assert json.loads((interval/"result.json").read_text())["rolling126_windows"]==819
src=dp["sources"];account_src=ip["source"]
features_card=root/src["features_card"];account=root/account_src["card"]
assert sha(features_card/"summary.json")==src["features_summary_sha256"]
assert sha(features_card/"nas_readback.json")==src["features_nas_readback_sha256"]
assert sha(account/"summary.json")==account_src["summary_sha256"]
fs=json.loads((features_card/"summary.json").read_text());fr=json.loads((features_card/"nas_readback.json").read_text());ac=json.loads((account/"summary.json").read_text())
assert fs["artifact_policy"]["recorder_archive_sha256"]==fr["NAS_archive_sha256"]==src["features_nas_archive_sha256"]
assert fs["nas_evidence"]["size_bytes"]==src["features_nas_size_bytes"]
assert ac["nas_readback"]["verified"] and ac["nas_readback"]["sha256"]==account_src["nas_sha256"]
fa=features_card/"qlib_recorder"/src["features_experiment_id"]/src["features_recorder_id"]/"artifacts"
aa=account/"qlib_recorder"/account_src["experiment_id"]/account_src["recorder_id"]/"artifacts"
assert sha(fa/"inputs/features")==src["features_artifact_sha256"]
assert sha(aa/"analysis/daily")==account_src["daily_sha256"]
assert sha(aa/"analysis/rolling126")==dep["account_rolling_sha256"]
assert sha(aa/"analysis/periods")==dep["account_periods_sha256"]
assert sha(root/src["provider"]/"qlib_store_manifest.json")==src["provider_manifest_sha256"]
qlib.init(provider_uri=str((root/src["provider"]).resolve()),region="us",expression_cache=None,dataset_cache=None,kernels=1)
with R.uri_context(str(features_card/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=src["features_experiment_id"],recorder_id=src["features_recorder_id"])
 features=rec.load_object("inputs/features")
with R.uri_context(str(account/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=account_src["experiment_id"],recorder_id=account_src["recorder_id"])
 daily=rec.load_object("analysis/daily").sort_values("trade_date").reset_index(drop=True)
 rolling=rec.load_object("analysis/rolling126")
 periods=rec.load_object("analysis/periods")
assert len(features)==3966 and len(daily)==944 and len(rolling)==819
assert len(periods.loc[periods.kind.eq("quarter")])==16
assert periods.loc[periods.kind.eq("year") & periods.partial_boundary_period,"period"].astype(str).tolist()==["2023","2026"]
original=fs["input_audit"]["original_features"]
assert len(original)==39 and list(features.columns[:39])==original
names=plan["fixed_inputs"]
assert len(names)==8 and len(set(names))==8 and set(names)<=set(original)
calendar=pd.DatetimeIndex(D.calendar(start_time="2011-01-03",end_time="2026-10-09",freq="day"))
close=D.features(["SOXX"],["$close"],start_time="2011-01-03",end_time="2026-10-09",freq="day")["$close"].droplevel("instrument").sort_index()
assert features.index.equals(calendar) and close.index.equals(calendar)
train=features.loc["2011-01-03":"2022-12-30",names]
held=features.reindex(pd.DatetimeIndex(daily.signal_date)).loc[:,names]
assert len(train)==3020 and len(held)==944 and held.notna().all().all()
median=train.median();sd=train.std(ddof=0)
assert np.isfinite(median.to_numpy()).all() and (sd>0).all()
z=held.sub(median).div(sd).reset_index(drop=True)
components=["cash_opportunity_log","SOXL_selection_log","Stage1_direct_fees_log","timing_and_benchmark_residual_log"]
assert float((daily[components].sum(axis=1)-daily.paired_log_excess).abs().max())<1e-12
windows=[];max_roll_error=0.0
for i in range(819):
 rw=rolling.iloc[i];g=daily.iloc[i:i+126]
 assert pd.Timestamp(g.trade_date.iloc[0])==pd.Timestamp(rw.first_trade)
 max_roll_error=max(max_roll_error,abs(float(g.paired_log_excess.sum())-float(rw.relative_log_excess)))
 signal=pd.Timestamp(g.signal_date.iloc[0]);loc=calendar.get_loc(signal);assert loc>=20
 r5=float(close.iloc[loc]/close.iloc[loc-5]-1);r20=float(close.iloc[loc]/close.iloc[loc-20]-1)
 state="uptrend" if r5>=0 and r20>=0 else "rebound" if r5>=0 else "pullback" if r20>=0 else "decline"
 row={"year":int(pd.Timestamp(rw.first_trade).year),"start_state":state,"outcome":"weak" if rw.relative_log_excess<0 else "good" if rw.relative_log_excess>0 else "tie"}
 row.update({name:float(z[name].iloc[i:i+126].mean()) for name in names})
 windows.append(row)
assert max_roll_error<1e-10
w=pd.DataFrame(windows)
assert w.outcome.value_counts().to_dict()=={"weak":564,"good":255}
cells=[]
for (state,year),g in w.groupby(["start_state","year"],sort=True):
 a=g.loc[g.outcome.eq("weak")];b=g.loc[g.outcome.eq("good")]
 item={"start_state":state,"year":int(year),"weak_windows":len(a),"good_windows":len(b),"matched":bool(len(a) and len(b))}
 if item["matched"]:item["weak_minus_good"]={name:float(a[name].mean()-b[name].mean()) for name in names}
 cells.append(item)
matched=[c for c in cells if c["matched"]]
assert len(matched)==12 and all(c["good_windows"]==0 for c in cells if c["year"]==2023)
weights=np.array([min(c["weak_windows"],c["good_windows"]) for c in matched],dtype=float)
weighted={name:float(np.average([c["weak_minus_good"][name] for c in matched],weights=weights)) for name in names}
signs={name:{"positive_cells":sum(c["weak_minus_good"][name]>0 for c in matched),"negative_cells":sum(c["weak_minus_good"][name]<0 for c in matched)} for name in names}
partial_2023={name:float(w.loc[w.year.eq(2023),name].mean()) for name in names}
result={"status":"read_only_eight_original_input_matched_interval_contrast_no_factor_selected",
        "plan_sha256":sha(plan_path),"input_recorder_id":src["features_recorder_id"],"account_recorder_id":account_src["recorder_id"],
        "train_sessions":len(train),"held_sessions":len(held),"rolling126_windows":len(w),"quarters":16,
        "matched_state_year_cells":len(matched),"unmatched_2023_cells":4,
        "fixed_inputs":names,"weighted_weak_minus_good_train_sd":weighted,"cell_signs":signs,"unmatched_2023_weak_window_mean_train_sd":partial_2023,"cells":cells,
        "new_market_requests":0,"new_model_fits":0,"new_native_accounts":0,"new_bootstraps":0,"new_recorders":0,
        "factor_selected":False,"policy_inference":False,"independent_validation":False,
        "limits":["Weak/good groups use future overlapping window outcomes and cannot define a trading switch",
                  "2023 has no same-year good interval control",
                  "Eight old input differences do not prove the model ignored them or that a new interaction improves SOXX net wealth"]}
(card/"result.json").write_text(json.dumps(result,ensure_ascii=False,indent=2)+"\n")
print({"weighted":weighted,"signs":signs,"2023_weak":partial_2023})


{'weighted': {'soxx_ret20': -0.12629309609560482, 'soxx_ret60': -0.2014229587922275, 'soxx_vol20': -0.06916635445660207, 'soxx_drawdown60': -0.030237234266285027, 'relative_momentum20': -0.19338074869504387, 'levered_decay_20': -0.3501151290139385, 'cross_breadth_pos_20': 0.07761592962053619, 'vix': 0.012360839061173861}, 'signs': {'soxx_ret20': {'positive_cells': 4, 'negative_cells': 8}, 'soxx_ret60': {'positive_cells': 3, 'negative_cells': 9}, 'soxx_vol20': {'positive_cells': 6, 'negative_cells': 6}, 'soxx_drawdown60': {'positive_cells': 5, 'negative_cells': 7}, 'relative_momentum20': {'positive_cells': 4, 'negative_cells': 8}, 'levered_decay_20': {'positive_cells': 3, 'negative_cells': 9}, 'cross_breadth_pos_20': {'positive_cells': 8, 'negative_cells': 4}, 'vix': {'positive_cells': 6, 'negative_cells': 6}}, '2023_weak': {'soxx_ret20': 0.254362528900163, 'soxx_ret60': 0.5219608865607165, 'soxx_vol20': 0.4340839506222773, 'soxx_drawdown60': -0.2542703106651781, 'relative_momentum20': 

## 中文结论

在读取评价区间前固定原39项中的八项：SOXX 20/60日收益、20日波动、60日回撤、20日SOXL相对动量、20日杠杆衰减、20日跨资产广度位置与VIX。只读封存Qlib输入及精确f696 A0 0bp Tiered原生账户；用2011–2022的3020个训练日确定中位数和标准差，再保留944个评价日、16季度、全部819个重叠126日窗口及两端不完整年度。

按窗口起点前已知 SOXX 5/20日趋势符号与首交易年配对，12格有好坏双方；2023四格全部没有好窗口。在这些可配对格子中，坏减好的训练标准差单位：SOXX60日收益 **−0.201**（9/12格为负）、相对动量20 **−0.193**（8/12负）、杠杆衰减20 **−0.350**（9/12负）。SOXX20日波动和VIX的格子正负均为6/6，未见同向区分。跨资产广度位置则有8/12格在坏窗口更高。全部八项逐格保存在 `result.json`，没有从评价收益选择阈值。

关键反例是2023：其全部126日窗口落后SOXX，但SOXX20/60日收益的窗口均值仍分别高于训练中位数约 **0.254/0.522** 个训练标准差。因而不能把后几年“坏窗口趋势较弱”的事后关联改写成统一的趋势开关；原f696已经接收这些输入却在2023留下现金机会成本。旧MA200、反弹确认与原39项接收器失败版本不因本卡重开。

结论 `descriptive_existing_trend_inputs_do_not_separate_2023_all_weak_from_later_matched_windows`。下一项可检验机制须事前区别强趋势但现金拖累的年份与真正有用的现金保护，并经过风险投影与完整账户；本卡只是提出问题，0拟合、0新账户，SOXX v2未达成。
